# 01 — Extraction Demo

Pipeline: **PDF → detect type → extract text blocks → filter noise**

This notebook demonstrates Layer 1 of the HSC-Edu pipeline on the textbook PDFs in `data/`.

In [10]:
from pathlib import Path

from hsc_edu.core.extraction import detect_pdf_type, extract_document

DATA_DIR = Path("../data")
PDF_FILES = sorted(DATA_DIR.glob("*.pdf"))

print(f"Found {len(PDF_FILES)} PDF(s):")
for p in PDF_FILES:
    print(f"  • {p.name}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")

Found 3 PDF(s):
  • C.pdf  (2.3 MB)
  • Java.pdf  (6.3 MB)
  • Python.pdf  (36.4 MB)


## 1. PDF Type Detection

Run `detect_pdf_type` on every PDF to check whether they are text-based, scanned, or mixed.

In [11]:
for pdf in PDF_FILES:
    r = detect_pdf_type(pdf)
    print(
        f"{pdf.name:<20s}  type={r.pdf_type.value:<12s}  "
        f"pages={r.total_pages}  text_pages={r.text_pages}  "
        f"scan_pages={r.scan_pages}  ratio={r.text_ratio:.2f}"
    )

C.pdf                 type=text-based    pages=102  text_pages=97  scan_pages=5  ratio=0.95
Java.pdf              type=text-based    pages=241  text_pages=231  scan_pages=10  ratio=0.96
Python.pdf            type=scanned       pages=219  text_pages=0  scan_pages=219  ratio=0.00


## 2. Text Block Extraction

Extract blocks from one PDF and inspect the results. Change `SAMPLE_PDF` to try a different file.

In [12]:
# Python.pdf is scanned (image-only) → use Java.pdf or C.pdf for text extraction demo
SAMPLE_PDF = DATA_DIR / "C.pdf"

blocks = extract_document(SAMPLE_PDF, doc_id="java-demo")
print(f"Total blocks extracted from '{SAMPLE_PDF.name}': {len(blocks)}")

Total blocks extracted from 'C.pdf': 2328


### 2.1 Statistics

In [13]:
from collections import Counter

pages = [b.page for b in blocks]
page_counts = Counter(pages)

num_pages = max(pages) + 1 if pages else 0
avg_blocks = len(blocks) / num_pages if num_pages else 0

font_sizes = [b.font_info.size for b in blocks if b.font_info]
bold_count = sum(1 for b in blocks if b.font_info and b.font_info.is_bold)

print(f"Pages covered        : {num_pages}")
print(f"Blocks per page (avg): {avg_blocks:.1f}")
print(f"Bold blocks          : {bold_count} / {len(blocks)}")
print(f"Font sizes found     : {sorted(set(round(s, 1) for s in font_sizes))}")

Pages covered        : 102
Blocks per page (avg): 22.8
Bold blocks          : 286 / 2328
Font sizes found     : [13.0, 16.0, 24.0, 37.0]


### 2.2 Blocks per page distribution

In [14]:
top_10 = page_counts.most_common(10)
print("Top 10 pages by block count:")
for pg, cnt in top_10:
    bar = "█" * cnt
    print(f"  Page {pg:>3d}: {cnt:>3d} blocks  {bar}")

Top 10 pages by block count:
  Page  27:  35 blocks  ███████████████████████████████████
  Page  66:  35 blocks  ███████████████████████████████████
  Page  68:  35 blocks  ███████████████████████████████████
  Page  71:  35 blocks  ███████████████████████████████████
  Page  73:  35 blocks  ███████████████████████████████████
  Page  78:  35 blocks  ███████████████████████████████████
  Page  43:  34 blocks  ██████████████████████████████████
  Page  63:  34 blocks  ██████████████████████████████████
  Page  65:  34 blocks  ██████████████████████████████████
  Page  67:  34 blocks  ██████████████████████████████████


## 3. Sample Blocks by Page

Print the first 10 blocks to inspect `raw_text`, `bbox`, and `font_info`.

In [15]:
SHOW_N = 10

for i, b in enumerate(blocks[:SHOW_N]):
    fi = b.font_info
    font_desc = f"{fi.name}, {fi.size}pt, bold={fi.is_bold}" if fi else "N/A"
    text_preview = b.raw_text[:120].replace("\n", " ↵ ")
    if len(b.raw_text) > 120:
        text_preview += " …"

    print(f"─── Block {i} ── page {b.page} ── bbox {b.bbox} ──")
    print(f"  Font : {font_desc}")
    print(f"  Text : {text_preview}")
    print()

─── Block 0 ── page 0 ── bbox (252.89, 362.23, 332.83, 395.47) ──
  Font : TimesNewRomanPS-BoldMT, 24.0pt, bold=True
  Text : BÀI GI

─── Block 1 ── page 0 ── bbox (329.59, 365.9, 350.11, 392.48) ──
  Font : TimesNewRomanPS-BoldMT, 24.0pt, bold=True
  Text : Ả

─── Block 2 ── page 0 ── bbox (346.99, 362.23, 386.23, 395.47) ──
  Font : TimesNewRomanPS-BoldMT, 24.0pt, bold=True
  Text : NG

─── Block 3 ── page 0 ── bbox (197.69, 397.28, 225.66, 448.5) ──
  Font : TimesNewRomanPS-BoldMT, 37.0pt, bold=True
  Text : L

─── Block 4 ── page 0 ── bbox (222.41, 402.93, 252.42, 443.89) ──
  Font : TimesNewRomanPS-BoldMT, 37.0pt, bold=True
  Text : Ậ

─── Block 5 ── page 0 ── bbox (249.05, 397.28, 405.56, 448.5) ──
  Font : TimesNewRomanPS-BoldMT, 37.0pt, bold=True
  Text : P TRÌNH

─── Block 6 ── page 0 ── bbox (411.55, 397.28, 441.59, 448.5) ──
  Font : TimesNewRomanPS-BoldMT, 37.0pt, bold=True
  Text : C

─── Block 7 ── page 1 ── bbox (271.13, 58.46, 355.18, 80.56) ──
  Font : TimesNewRomanPS-

## 4. Inspect a Specific Page

Pick a page to see all its blocks in detail (useful for checking heading detection later).

In [18]:
TARGET_PAGE = 15  # ← change this to inspect another page

page_blocks = [b for b in blocks if b.page == TARGET_PAGE]
print(f"Page {TARGET_PAGE}: {len(page_blocks)} block(s)\n")

for i, b in enumerate(page_blocks):
    fi = b.font_info
    font_desc = f"{fi.name}, {fi.size}pt, bold={fi.is_bold}" if fi else "N/A"

    print(f"[{i}] bbox={b.bbox}  font=({font_desc})")
    print(f"    {b.raw_text}")
    print()

Page 15: 25 block(s)

[0] bbox=(89.42, 72.92, 532.49, 90.59)  font=(TimesNewRomanPS-BoldMT, 16.0pt, bold=True)
    Chương 2: LÀM QUEN LẬP TRÌNH C QUA CÁC VÍ DỤ ĐƠN

[1] bbox=(288.53, 90.52, 338.38, 112.62)  font=(TimesNewRomanPS-BoldMT, 16.0pt, bold=True)
    GIẢN

[2] bbox=(71.54, 151.73, 258.89, 169.8)  font=(TimesNewRomanPS-BoldMT, 13.0pt, bold=True)
    2.1. Cài đặt và sử dụng Dev-C++

[3] bbox=(71.54, 171.65, 555.78, 262.26)  font=(TimesNewRomanPSMT, 13.0pt, bold=False)
    2.1.1. Giới thiệu về Dev-C++  
 Dev-C++ (hay còn gọi Dev-Cpp) là một bộ công cụ phát triển tích hợp (IDE Integrated 
Development Environment) các ứng dụng C/C++ thuộc dạng mã nguồn mở và có thể  
download (gõ download Dev-C++ vào google). DevCpp sử dụng cho cả hệ thống Windows 
và Linux. Hiện nay DevCpp  là công cụ phát triển các ứng dụng C/C++ được sử dụng rộng

[4] bbox=(72.26, 265.28, 470.5, 282.57)  font=(TimesNewRomanPSMT, 13.0pt, bold=False)
    rãi để dạy về lập trình cũng như để phát  triển các ứng dụng